# Context-Aware Dream Interpretation System

## Project Goal

Build a system that summarizes a dream and generates tentative, evidence-grounded theories about its themes. The system can retrieve related past dreams based on recurring people, locations, setting types, events, and emotions.

## Research Questions

1. Does QLoRA improve structured extraction of people, locations, settings, events, and emotions compared with an unchanged base model?

2. Does RAG make dream theories more personally relevant and grounded than analyzing the current dream alone, or can retrieval reduce quality?



In [4]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu128
GPU available: True
GPU: Tesla T4


In [6]:
!pip install -q transformers datasets peft trl bitsandbytes accelerate sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 97.3 MB/s eta 0:00:00


In [7]:
import random
import numpy as np
import torch

from datasets import Dataset
from transformers import set_seed

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

print("Setup complete.")

Setup complete.


## RAG Corpus: Load Private Dream Entries

In [8]:
import os

DATA_FOLDER = "/content/drive/MyDrive/IT344/DJournal Project/private_data"

print(os.listdir(DATA_FOLDER))

['dream1.txt', 'dream2.txt', 'dream3.txt', 'dream4.txt', 'dream5.txt', 'dream6.txt', 'dream7.txt', 'dream8.txt', 'dream9.txt', 'dream10.txt', 'dream11.txt', 'dream12.txt', 'dream13.txt', 'dream14.txt', 'dream15.txt']


In [9]:
from pathlib import Path
import re

def dream_number(path):
    match = re.search(r"\d+", path.stem)
    return int(match.group()) if match else 999999

dream_files = sorted(
    Path(DATA_FOLDER).glob("*.txt"),
    key=dream_number
)

dream_entries = []

for file_path in dream_files:
    dream_entries.append({
        "dream_id": file_path.stem,
        "text": file_path.read_text(encoding="utf-8").strip()
    })

print("Dreams loaded:", len(dream_entries))
print("Files:", [dream["dream_id"] for dream in dream_entries])

Dreams loaded: 15
Files: ['dream1', 'dream2', 'dream3', 'dream4', 'dream5', 'dream6', 'dream7', 'dream8', 'dream9', 'dream10', 'dream11', 'dream12', 'dream13', 'dream14', 'dream15']


## Create Dream Embeddings


In [10]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = "all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(EMBEDDING_MODEL)

dream_texts = [dream["text"] for dream in dream_entries]

dream_embeddings = embedding_model.encode(
    dream_texts,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding shape:", dream_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding shape: (15, 384)


## Build the Retrieval Index

In [11]:
import faiss

embedding_dimension = dream_embeddings.shape[1]

rag_index = faiss.IndexFlatIP(embedding_dimension)
rag_index.add(dream_embeddings.astype("float32"))

print("Dreams indexed:", rag_index.ntotal)

Dreams indexed: 15


## Dream Retrieval Function


In [12]:
def retrieve_similar_dreams(query_text, top_k=3):
    query_embedding = embedding_model.encode(
        [query_text],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = rag_index.search(query_embedding, top_k)

    results = []

    for score, index in zip(scores[0], indices[0]):
        results.append({
            "dream_id": dream_entries[index]["dream_id"],
            "similarity_score": float(score),
            "text": dream_entries[index]["text"]
        })

    return results

print("Retrieval function ready.")

Retrieval function ready.


In [13]:
test_dream = """
I was in a big city, it felt like Chicago. There were tall and wide brick buildings. This was like an isolated town, you'd drive
long desolate roads before suddenly reaching this burst of life in this pocket of an area. There were tons of food stands to the right
and I was so excited. But something felt off. Something felt scary...vulnerable. Eventually it got dark, and suddenly I was running
as fast as I could with the brick buildings to the left of me, with my direction heading back from the city and toward the entrance where
my car was parked. My car felt out of place and vulnerable. I felt like I had a lot of attention on me."""

retrieved_dreams = retrieve_similar_dreams(test_dream, top_k=3)

for result in retrieved_dreams:
    print(
        result["dream_id"],
        "- similarity:",
        round(result["similarity_score"], 3)
    )

dream11 - similarity: 0.532
dream4 - similarity: 0.527
dream3 - similarity: 0.508


## Unchanged Base Model

In [14]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    attn_implementation="eager"
)

print("Base model loaded.")

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Base model loaded.


## Current Dream Only


In [15]:
def generate_baseline(dream_text):
    messages = [
        {
            "role": "user",
            "content": (
                "Analyze and summarize this dream. Identify its important people, places, "
                "events, emotions, atmosphere, prominent elements, and easily overlooked details. "
                "Then explore possible interpretations through past-life continuity, parallel "
                "realities, etheric or nonlocal information, energetic resonance, synchronicity, "
                "collective fields, field interference, temporal echoes, place memory, astral "
                "experience, and collective thought-forms. Connect each possibility to details "
                "actually present in the dream. Be deeply open-minded, but clearly treat every "
                "interpretation as speculative rather than proven fact.\n\nDream:\n" + dream_text
            )
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(base_model.device)

    with torch.no_grad():
        outputs = base_model.generate(
            **inputs,
            max_new_tokens=1200,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

baseline_output = generate_baseline(test_dream)

print(baseline_output)

**Dream Analysis Summary:**

**Important People:**
- The dreamer is the central figure, experieniously running and feeling vulnerable.

**Places:**
- A big city resembling Chicago with isolated pockets of life amidst desolate roads.
- A food stand area with excitement.
- A parking lot where the dreamer's car is parked.

**Events:**
- The dreamer is running away from the city, heading back to the parking lot.

**Emotions:**
- Excitement at the food stands.
- Fear and vulnerability as the dreamer runs away from the city.
- Attention from an unspecified source.

**Atmosphere:**
- The city feels isolated and desolate, with a sudden burst of life in certain areas.
- The atmosphere shifts from excitement to fear and vulnerability.

**Prominent Elements:**
- Tall and wide brick buildings.
- Food stands.
- A car that feels out of place and vulnerable.

**Overlooked Details:**
- The specific reason for the feeling of vulnerability and the attention from others.
- The nature of the "burst of lif

In [19]:
def generate_prompt_v1(dream_text):
    messages = [
        {
            "role": "user",
            "content": (
                "Perform a deep, open-minded analysis of the dream below. "
                "Treat all metaphysical interpretations as speculative lenses, not proven facts. Summary at the end brief. "
                "Do not invent, alter, or exaggerate any dream detail.\n\n"

                "Begin with SPECULATIVE LENSES so this section is never omitted. "
                "Consider past-life continuity, parallel or alternate realities, "
                "etheric or nonlocal information, energetic resonance, synchronicity, "
                "collective fields, field interference, temporal echoes, place imprints, "
                "astral or out-of-body experience, and collective thought-forms.\n\n"

                "For every relevant lens:\n"
                "- Name the lens.\n"
                "- Explain how it could interpret the dream.\n"
                "- Quote or closely paraphrase the exact dream details supporting it.\n"
                "- Rate the feature match as strongly, moderately, or weakly activated.\n"
                "- Clearly state what information is missing or contradictory.\n\n"

                "Afterward, provide:\n"
                "1. A brief summary of what happened.\n"
                "2. The most prominent elements and why they are prominent.\n"
                "3. People, exact locations, and vague setting types.\n"
                "4. Important events, emotions, sensations, and atmosphere.\n"
                "5. Small or passing details that should not be forgotten.\n"
                "6. Ordinary alternatives such as memory recombination, recent exposure, "
                "expectation, coincidence, emotional priming, or source confusion.\n\n"
"Treat these as speculative interpretations, not established facts.\n\n"

                "Use polished spelling and complete sentences. Separate stated dream facts "
                "from interpretations. "
                "Describe the writer as the title dreamer in the texts. Theory only.\n\n"

                "Dream:\n" + dream_text
            )
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(base_model.device)

    with torch.no_grad():
        outputs = base_model.generate(
            **inputs,
            max_new_tokens=1200,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

prompt_v1_output = generate_prompt_v1(test_dream)

print(prompt_v1_output)

### Speculative Interpretations Based on Metaphysical Lenses

#### Past-Life Continuity (Weak Activation)
The feeling of being "out of place" might suggest memories or experiences related to urban environments from previous lives when one may have been more attuned to nature rather than cities. However, there isn't enough specific evidence within the dream content itself to support strong activation for past-life connections without additional personal history data.

#### Parallel Realities/Alternate Reality (Moderate Activation)
In some theories about multiple dimensions, our current reality can intersect briefly with others through quantum entanglement—a concept which seems reflected here by experiencing both excitement towards abundant resources ("tons of food stands") while simultaneously facing fear due to isolation amidst vastness. Yet again, direct proof linking the dream scenario specifically to alternative realities remains unsubstantiated based solely on the narrative provide

### Prompt Version 2: Structured Analysis Without RAG

In [20]:
def generate_prompt_v2(dream_text):

    messages = [
        {
            "role": "system",
            "content": (
                "You are an advanced dream-pattern analyst. "
                "Analyze dreams with deep curiosity and radical open-mindedness. "
                "Consider conventional explanations and speculative possibilities, "
                "including past-life continuity, parallel or alternate realities, "
                "shared or telepathic dreams, precognitive dreams, synchronicity, "
                "nonlocal consciousness, collective fields, energetic or morphic resonance, "

                "Do not dismiss these possibilities merely because they are unconventional, "
                "but never present them as proven facts. Clearly distinguish dream evidence, "
                "possible interpretations, and ordinary alternative explanations. "
                "Do not make generic psychological claims about what the dreamer secretly wants, "
                "needs, fears, or is searching for in life."
            )
        },
        {
            "role": "user",
            "content": (
                "Analyze the following dream carefully and in depth.\n\n"
                f"DREAM:\n{dream_text}\n\n"

                "Follow this exact structure:\n\n"

                "1. DREAM RECORD\n"
                "Briefly reconstruct what happened using only information explicitly present "
                "in the dream. Do not add events or details.\n\n"

                "2. PROMINENT ELEMENTS\n"
                "Identify the strongest locations, people, objects, actions, atmosphere, "
                "emotions, bodily sensations, transitions, and unusual details. Explain which "
                "elements receive the most description or emotional intensity.\n\n"

                "3. SMALL DETAILS THAT SHOULD NOT BE LOST\n"
                "Preserve details mentioned only briefly that could become important when "
                "compared with future or previous dreams.\n\n"



                "For each selected lens:\n"
                "- Name the lens.\n"
                "- Identify the exact dream features that activated it.\n"
                "- Explain the possible relationship in detail.\n"
                "- State what additional evidence would strengthen it.\n"
                "- State what evidence would weaken or contradict it.\n"
                "- Give an ordinary process that could create a similar experience.\n"
                "- Suggest an observation that could help distinguish the possibilities.\n\n"

                "5. OTHER POSSIBLE LENSES\n"
                "Briefly identify any additional speculative, cultural, spiritual, cognitive, "
                "memory-based, or emotional frameworks that may be relevant.\n\n"

                "6. INTERFERENCE AND ALTERNATIVE EXPLANATIONS\n"
                "Consider memory incorporation, expectation, coincidence, false memory, source "
                "contamination, retrospective matching, recent experiences, sleep paralysis, "
                "ordinary dream construction, or other processes that could imitate an unusual "
                "connection. Do not automatically treat these explanations as superior; compare "
                "them fairly with the speculative possibilities.\n\n"

                "7. FINAL SYNTHESIS\n"
                "Summarize the dream's strongest patterns and most unusual relationships. "
                "State which possibilities appear most relevant and why, while clearly marking "
                "uncertainty. End with useful details or patterns to watch for in later dreams.\n\n"

                "Do not claim that any theory is proven. Do not diagnose the dreamer. "
                "Do not invent missing information. Be detailed, specific, and genuinely "
                "open-minded rather than giving a generic dream interpretation."
            )
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(base_model.device)

    with torch.no_grad():
        outputs = base_model.generate(
            **inputs,
            max_new_tokens=1200,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response


prompt_v2_output = generate_prompt_v2(test_dream)

print(prompt_v2_output)

1. **DREAM RECORD**
The individual recalls being in a large urban setting reminiscent of Chicago, characterized by expansive brick structures forming both streets and walls around their pathway. The journey through seemingly endless roadways leads abruptly into bustling activity within a confined space filled with numerous eateries situated prominently to one side. As night falls, there is a palpable sense of unease coupled with urgency propelling the person away from the dense cluster towards their vehicle located at the periphery—a feeling amplified due to its incongruous placement amidst such surroundings.

2. **PROMINENT ELEMENTS**
   - Urban Environment (Chicago): Activates "City Dream" lens suggesting familiarization with metropolitan settings possibly indicating subconscious connections to personal history related to cities. Additional evidence might include frequent visits or significant memories tied to particular places. Contradictory evidence includes no prior mention of lon